# Agentic Debugging QLoRA Patch Pilot v1 — Final Training Run

Authorized scope: **final training only** (one bounded, descriptive one-epoch QLoRA run on the
frozen 1,000-train / 150-validation corpus). Held-out generation and base-versus-tuned evaluation
remain **unauthorized**; no held-out task content is loaded anywhere in this notebook. This notebook
is executed by the owner in Colab after FirstMate review; it does not rebuild or top up the corpus.


In [ ]:
# Cell 1 — Install frozen user-space dependencies. Do not pin Colab's CUDA torch wheel.
%pip install -q \
  transformers==5.14.1 \
  datasets==5.0.0 \
  peft==0.20.0 \
  trl==1.8.0 \
  bitsandbytes==0.49.2 \
  accelerate==1.14.0 \
  huggingface_hub \
  safetensors


In [ ]:
# Cell 2 — Mount persistent external storage and identify the repository.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
REPOSITORY_ROOT = Path('/content/agentic-debugging-internship')
DRIVE_ROOT = Path('/content/drive/MyDrive/agentic-debugging/qlora_patch_pilot_v1')
CORPUS_ROOT = DRIVE_ROOT / 'corpus'
FINAL_ROOT = DRIVE_ROOT / 'final-training'
MODEL_CACHE = DRIVE_ROOT / 'model-cache'
for path in (FINAL_ROOT, MODEL_CACHE):
    path.mkdir(parents=True, exist_ok=True)
assert (REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/freeze_record.json').is_file()
assert (CORPUS_ROOT / 'train.jsonl').is_file() and (CORPUS_ROOT / 'validation.jsonl').is_file()
%cd {REPOSITORY_ROOT}


In [ ]:
# Cell 3 — Verify all frozen local identities before data or model work.
import json, subprocess, sys
FREEZE = REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/freeze_record.json'
result = subprocess.run([
    sys.executable, 'scripts/qlora_patch_pilot.py', 'verify-freeze',
    '--repository-root', str(REPOSITORY_ROOT), '--freeze-record', str(FREEZE),
], check=True, text=True, capture_output=True)
verification = json.loads(result.stdout)
freeze = json.loads(FREEZE.read_text(encoding='utf-8'))
assert verification['status'] == 'LOCKED' and verification['failed'] == []
assert freeze['repository_baseline']['base_commit'] == '66fb5d5'
assert freeze['repository_baseline']['relationship'] == 'required_ancestor'
assert freeze['scientific_gate']['final_training_authorized'] is False
assert freeze['scientific_gate']['held_out_generation_authorized'] is False
print(json.dumps(verification['runtime'], indent=2))


In [ ]:
# Cell 4 — Validate the separate final-training authorization record (fail closed).
AUTHORIZATION = REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/final_training_authorization.json'
assert AUTHORIZATION.is_file(), f'final-training authorization record missing: {AUTHORIZATION}'
auth_command = [
    sys.executable, 'scripts/qlora_patch_pilot.py', 'validate-final-training-auth',
    '--authorization', str(AUTHORIZATION),
    '--repository-root', str(REPOSITORY_ROOT),
    '--corpus-dir', str(CORPUS_ROOT),
]
result = subprocess.run(auth_command, check=True, text=True, capture_output=True)
auth_result = json.loads(result.stdout)
assert auth_result['status'] == 'COMPLETE'
assert auth_result['authorization_scope'] == 'final_training_only'
assert auth_result['authorized'] is True
assert auth_result['held_out_generation_authorized'] is False
assert auth_result['base_versus_tuned_evaluation_authorized'] is False
print(result.stdout)


In [ ]:
# Cell 5 — Re-run the fail-closed independent audit validation before model load.
COMPLETED_AUDIT = DRIVE_ROOT / 'independent-audit' / 'firstmate_independent_audit_completed.csv'
assert COMPLETED_AUDIT.is_file(), f'completed independent audit CSV missing: {COMPLETED_AUDIT}'
audit_command = [
    sys.executable, 'scripts/qlora_patch_pilot.py', 'validate-audits',
    '--output-dir', str(CORPUS_ROOT),
    '--transformation-config', str(REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/transformation_config.json'),
    '--completed-audit', str(COMPLETED_AUDIT),
]
result = subprocess.run(audit_command, check=True, text=True, capture_output=True)
audit_result = json.loads(result.stdout)
assert audit_result['status'] == 'COMPLETE'
assert audit_result['audit_mode'] == 'independent_ai'
assert audit_result['accepted_packet_accept'] == 39
assert audit_result['accepted_packet_reject'] == 11
assert audit_result['accepted_packet_total'] == 50
assert audit_result['rejected_packet_accept'] == 0
assert audit_result['rejected_packet_reject'] == 25
assert audit_result['rejected_packet_total'] == 25
print(result.stdout)


In [ ]:
# Cell 6 — Verify the frozen corpus records; never rebuild or top up.
corpus_summary = json.loads((CORPUS_ROOT / 'corpus_summary.json').read_text(encoding='utf-8'))
dedup_report = json.loads((CORPUS_ROOT / 'dedup_report.json').read_text(encoding='utf-8'))
assert corpus_summary['corpus_tier'] == 'minimum'
assert corpus_summary['train_examples'] == 1000
assert corpus_summary['validation_examples'] == 150
assert dedup_report['repository_overlap'] == []
assert dedup_report['held_out_exact_matches_accepted'] == 0
assert dedup_report['held_out_near_matches_accepted'] == 0
print(corpus_summary)


In [ ]:
# Cell 7 — Record runtime identity and require a CUDA accelerator.
import importlib.metadata as md, platform, time
import torch
runtime_started = time.time()
runtime = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_runtime': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_total_memory_bytes': torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
    'packages': {name: md.version(name) for name in ['transformers','datasets','peft','trl','bitsandbytes','accelerate','huggingface_hub','safetensors']},
    'repository_verification': verification['runtime'],
}
(FINAL_ROOT / 'runtime_environment.json').write_text(json.dumps(runtime, indent=2, sort_keys=True) + '\n')
print(json.dumps(runtime, indent=2))
assert torch.cuda.is_available(), 'A CUDA Colab runtime is required for the final training run.'


In [ ]:
# Cell 8 — Load the frozen tokenizer and train/validation files (no held-out content).
from datasets import load_dataset
from transformers import AutoTokenizer
training_cfg = json.loads((REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/training_config.json').read_text())
model_id = training_cfg['model_repository']
model_revision = training_cfg['model_revision']
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=model_revision, cache_dir=str(MODEL_CACHE), use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
raw_train = load_dataset('json', data_files=str(CORPUS_ROOT / 'train.jsonl'), split='train')
raw_validation = load_dataset('json', data_files=str(CORPUS_ROOT / 'validation.jsonl'), split='train')

def tokenize_completion_only(example):
    prompt_text = tokenizer.apply_chat_template(example['prompt'], tokenize=False, add_generation_prompt=True)
    completion = example['completion']
    full_text = prompt_text + completion + (tokenizer.eos_token or '')
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
    encoded = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=training_cfg['sft']['max_length'])
    labels = [-100] * min(len(prompt_ids), len(encoded['input_ids'])) + encoded['input_ids'][len(prompt_ids):]
    encoded['labels'] = labels
    return encoded

tokenized_train = raw_train.map(tokenize_completion_only, remove_columns=raw_train.column_names)
tokenized_validation = raw_validation.map(tokenize_completion_only, remove_columns=raw_validation.column_names)
assert len(tokenized_train) == 1000 and len(tokenized_validation) == 150
print({'train_examples': len(tokenized_train), 'validation_examples': len(tokenized_validation)})


In [ ]:
# Cell 9 — Load the pinned 7B checkpoint in frozen 4-bit QLoRA configuration.
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant = training_cfg['quantization']
quant_config = BitsAndBytesConfig(
    load_in_4bit=quant['load_in_4bit'],
    bnb_4bit_quant_type=quant['quant_type'],
    bnb_4bit_use_double_quant=quant['double_quant'],
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=model_revision, quantization_config=quant_config,
    device_map={'': 0}, cache_dir=str(MODEL_CACHE), dtype=compute_dtype,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_cfg = training_cfg['lora']
lora = LoraConfig(
    r=lora_cfg['r'], lora_alpha=lora_cfg['alpha'], lora_dropout=lora_cfg['dropout'],
    target_modules=lora_cfg['target_modules'], bias=lora_cfg['bias'], task_type=lora_cfg['task_type'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
# Cell 10 — Train exactly one epoch with the frozen training configuration.
from dataclasses import dataclass
from transformers import Trainer, TrainingArguments

@dataclass
class CompletionOnlyCollator:
    pad_token_id: int
    def __call__(self, features):
        max_len = max(len(item['input_ids']) for item in features)
        batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
        for item in features:
            pad = max_len - len(item['input_ids'])
            batch['input_ids'].append(item['input_ids'] + [self.pad_token_id] * pad)
            batch['attention_mask'].append(item['attention_mask'] + [0] * pad)
            batch['labels'].append(item['labels'] + [-100] * pad)
        return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}

sft = training_cfg['sft']
final_adapter = FINAL_ROOT / 'adapter-final'
args = TrainingArguments(
    output_dir=str(FINAL_ROOT / 'trainer-output'),
    per_device_train_batch_size=sft['per_device_train_batch_size'],
    gradient_accumulation_steps=sft['gradient_accumulation_steps'],
    num_train_epochs=sft['num_train_epochs'],
    learning_rate=sft['learning_rate'],
    optim=sft['optim'],
    lr_scheduler_type=sft['lr_scheduler_type'],
    warmup_ratio=sft['warmup_ratio'],
    logging_steps=10,
    save_strategy='no',
    report_to='none',
    fp16=compute_dtype == torch.float16,
    bf16=compute_dtype == torch.bfloat16,
    seed=sft['seed'],
    data_seed=sft['data_seed'],
    gradient_checkpointing=sft['gradient_checkpointing'],
    gradient_checkpointing_kwargs={'use_reentrant': False},
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized_train, eval_dataset=tokenized_validation,
    data_collator=CompletionOnlyCollator(tokenizer.pad_token_id),
)
train_started = time.time()
train_result = trainer.train()
elapsed_seconds = time.time() - train_started
peak_memory = {
    'peak_cuda_memory_allocated_bytes': torch.cuda.max_memory_allocated(),
    'peak_cuda_memory_reserved_bytes': torch.cuda.max_memory_reserved(),
}
print({'train_loss': train_result.training_loss, 'elapsed_seconds': elapsed_seconds, **peak_memory})


In [ ]:
# Cell 11 — Save adapter, tokenizer, trainer state, logs, summary, sizes and checksums.
from agentic_debugger.training.patch_pilot import sha256_bytes, write_external_manifest
model.save_pretrained(final_adapter, safe_serialization=True)
tokenizer.save_pretrained(final_adapter)
trainer_state = {
    'global_step': trainer.state.global_step,
    'epoch': trainer.state.epoch,
    'log_history': trainer.state.log_history,
}
(FINAL_ROOT / 'trainer_state.json').write_text(json.dumps(trainer_state, indent=2, sort_keys=True) + '\n')
(FINAL_ROOT / 'training_log_history.json').write_text(json.dumps(trainer.state.log_history, indent=2, sort_keys=True) + '\n')
artifact_hashes = {}
for path in sorted(final_adapter.rglob('*')):
    if path.is_file():
        artifact_hashes[str(path.relative_to(final_adapter))] = sha256_bytes(path.read_bytes())
training_summary = {
    'schema_version': 'final-training-summary-v1',
    'experiment_id': 'qlora-patch-pilot-v1',
    'authorized': True,
    'authorization_scope': 'final_training_only',
    'held_out_generation_authorized': False,
    'train_loss': train_result.training_loss,
    'elapsed_seconds': elapsed_seconds,
    **peak_memory,
    'train_examples': 1000,
    'validation_examples': 150,
    'epochs': sft['num_train_epochs'],
    'trainer_state': trainer_state,
    'runtime': runtime,
    'adapter_files_sha256': artifact_hashes,
}
(FINAL_ROOT / 'final_training_summary.json').write_text(json.dumps(training_summary, indent=2, sort_keys=True) + '\n')
manifest = write_external_manifest(
    FINAL_ROOT,
    configuration_identity=freeze['training']['sha256'],
    provenance_identity=f"{model_id}@{model_revision}",
    artifact_kind_prefix='final-training',
)
print(json.dumps(manifest, indent=2)[:4000])


In [ ]:
# Cell 12 — Reload the saved final adapter and verify it is usable.
import gc
from peft import PeftModel
del trainer, model
gc.collect(); torch.cuda.empty_cache()
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=model_revision, quantization_config=quant_config,
    device_map={'': 0}, cache_dir=str(MODEL_CACHE), dtype=compute_dtype,
)
reloaded_final = PeftModel.from_pretrained(base_model, final_adapter, is_trainable=False)
reloaded_final.eval()
print({'adapter_reloaded': True, 'adapter_path': str(final_adapter)})


In [ ]:
# Cell 13 — Hard gate: held-out generation remains unauthorized. Stop here.
assert auth_result['held_out_generation_authorized'] is False
assert freeze['scientific_gate']['held_out_generation_authorized'] is False
print('FINAL_TRAINING_COMPLETE_AWAITING_FIRSTMATE_REVIEW')


## Boundary — do not continue past this cell

No held-out task content was loaded or generated by this notebook. Held-out generation,
base-versus-tuned evaluation, dataset acquisition, and corpus modification remain forbidden
until FirstMate reviews the final adapter and `final_training_summary.json`. The adapter and
all records above are on Drive under `agentic-debugging/qlora_patch_pilot_v1/final-training/`.
